In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [3]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [4]:
documents = documents_llm

In [5]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [6]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [7]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [9]:
import json
user_prompt = json.dumps(doc)

In [10]:
user_prompt

'{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General Course-Related Questions", "question": "I just discovered the course. Can I still join?", "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."}'

In [11]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [12]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [13]:
response.output_parsed.questions

['I just found this course — am I still allowed to join now, or is it too late?',
 'If I start the course late, can I still get a certificate somehow?',
 'Is it okay to join after the course has already started, or do I have to wait for the next run?',
 'What do I need to do if I want a certificate after joining the course late?',
 'Can I still submit the project for certification if I discovered the course after it began?']

In [14]:
doc

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [15]:
from evaluation_utils import llm_structured

In [16]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course — is it still okay to join now, or am I too late?', 'Can I enroll after the course has already started and still follow along?', 'If I join late, is it still possible to get a certificate for the course?', 'What’s the deadline for the final project if I want the certificate?', 'Do I need to submit my project before submissions close in order to get certified?']


In [17]:
usage

ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=97, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=304)

In [18]:
from evaluation_utils import calc_price

In [19]:
calc_price(usage)

{'input_cost': 0.00015525, 'output_cost': 0.0004365, 'total_cost': 0.00059175}

In [20]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course — is it still okay to join now, or am I too late?',
  'document': '74eb249bbf'},
 {'question': 'Can I enroll after the course has already started and still follow along?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, is it still possible to get a certificate for the course?',
  'document': '74eb249bbf'},
 {'question': 'What’s the deadline for the final project if I want the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to submit my project before submissions close in order to get certified?',
  'document': '74eb249bbf'}]

In [21]:
import pandas as pd

In [22]:
pd.DataFrame(records)

,question,document
0,I just found this course — is it still okay to...,74eb249bbf
1,Can I enroll after the course has already star...,74eb249bbf
2,"If I join late, is it still possible to get a ...",74eb249bbf
3,What’s the deadline for the final project if I...,74eb249bbf
4,Do I need to submit my project before submissi...,74eb249bbf


In [23]:
from evaluation_utils import llm_structured_retry

In [25]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [26]:
generate_ground_truth(doc)

([{'question': 'I found this course late — can I still join, or is it already closed?',
   'document': '74eb249bbf'},
  {'question': 'If I start the course now, can I still get a certificate somehow?',
   'document': '74eb249bbf'},
  {'question': 'Is it okay to join the course after it has already started?',
   'document': '74eb249bbf'},
  {'question': 'What do I need to do to be eligible for the certificate if I’m joining late?',
   'document': '74eb249bbf'},
  {'question': 'Are late joiners allowed, and does the project deadline affect the certificate?',
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=94, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=301))

In [27]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [28]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [29]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

In [30]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

565

In [31]:
ground_truth[10]

{'question': 'Where do I watch the Office Hours or live workshop stream if I’m a student?',
 'document': '489dd1c9d9'}

In [32]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08790449999999995

In [33]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.08790449999999995

In [34]:
df_ground_truth = pd.DataFrame(ground_truth)

In [36]:
df_ground_truth.to_csv("../data/ground_truth.csv", index=False)

In [37]:
len(df_ground_truth)

565